In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import os
import re
import os
import gdown





File already exists at ../data/OA_All_Census_21.parquet
Columns with missing values:
['ts0200002_PCT', 'ts0200003_PCT', 'ts0550002_PCT', 'ts0550003_PCT', 'ts0550004_PCT', 'ts0550005_PCT', 'ts0550006_PCT', 'ts0550007_PCT', 'ts0550008_PCT', 'ts0550009_PCT', 'ts0550010_PCT']


In [ ]:


# Download the Parquet file from Google Drive
file_id = '1F_vFJfWon15iWZw6l35smt5Jyb2aPeCn'
output = 'OA_All_Census_21.parquet'
# Define the target path
data_path = f"../data/{output}"
# Check if the file already exists
if not os.path.exists(data_path):
    # Download the Parquet file from Google Drive
    gdown.download(f'https://drive.google.com/uc?id={file_id}', data_path, quiet=False)
else:
    print(f"File already exists at {data_path}")

# Read the 2021 census parquet file into a DataFrame
All_census = pd.read_parquet(data_path)
#set OA as index
All_census = All_census.set_index('OA')

# Drop the column 'ts0060001' and store in a new DataFrame called 'W' (weight)
All_W = All_census['ts0060001']
All_census = All_census.drop(columns=['ts0060001'])

# Remove the Wales only variables

# Strings to search for
strings_to_remove = ['ts032', 'ts033', 'ts034', 'ts035', 'ts036', 'ts076']
# Filter columns
columns_to_keep = [col for col in All_census.columns if not any(string in col for string in strings_to_remove)]

# Filter the DataFrame to keep only the desired columns
All_census = All_census[columns_to_keep]

#print columns with missing values
print("Columns with missing values:")
missing_columns = All_census.columns[All_census.isnull().any()].tolist()
print(missing_columns)
# All_census = All_census.fillna(0)

All_census.columns = All_census.columns.str.replace('_PCT', '', regex=False)

In [8]:

def populate_data_table(filepath: str) -> pd.DataFrame:
    """
    Populates a data table by merging CSV files in the specified directory.

    Args:
        filepath (str): The path to the directory containing the CSV files.

    Returns:
        pd.DataFrame: The merged data table, or None if no files are found.
    """
    print("Populating data table from files in", filepath)
    data_table = None  # Initialize data_table outside the loop
    # Divide all columns but the first by the second column
    for file in tqdm(os.listdir(filepath), desc="Processing files", unit="file"):
        df = pd.read_csv(filepath + "/" + file, index_col="OA")

        if data_table is None:
            data_table = df  # Initialize data_table with the first file encountered
        else:
            # Merge on index
            data_table = data_table.merge(df, left_index=True, right_index=True, how="outer")
    return data_table

eng_census_raw = populate_data_table("../data/census_data/eng_raw_csvs")
#sort the columns
eng_census_raw = eng_census_raw.reindex(sorted(eng_census_raw.columns), axis=1)

eng_census_raw = eng_census_raw.drop(columns=['ts0060001'])
# Remove the Wales only variables
# Strings to search for
strings_to_remove = ['ts032', 'ts033', 'ts034', 'ts035', 'ts036', 'ts076']
# Filter columns
columns_to_keep = [col for col in eng_census_raw.columns if not any(string in col for string in strings_to_remove)]
# Filter the DataFrame to keep only the desired columns
eng_census_raw = eng_census_raw[columns_to_keep]

# Extract unique prefixes by removing the last four digits
prefixes = set(re.sub(r"\d{4}$", "", col) for col in eng_census_raw.columns if re.match(r".*\d{4}$", col))


# Normalize each group
for prefix in prefixes:
    print(f"Normalizing columns with prefix: {prefix}")

    base_col = f"{prefix}0001"  # The assumed total column
    group_cols = [col for col in eng_census_raw.columns if col.startswith(prefix)]

    if base_col in eng_census_raw.columns:  # Ensure the base column exists
        eng_census_raw[group_cols] = eng_census_raw[group_cols].div(eng_census_raw[base_col], axis=0)
#drop the total columns (anything ending 001)

# Filter columns that end with '001'
columns_to_drop = [col for col in eng_census_raw.columns if col.endswith('001')]
print("Dropping columns ending with '001':", columns_to_drop)
# Drop the columns
eng_census_raw = eng_census_raw.drop(columns=columns_to_drop)


Populating data table from files in ../data/census_data/eng_raw_csvs


Processing files: 100%|██████████| 52/52 [00:17<00:00,  3.01file/s]


Normalizing columns with prefix: ts050
Normalizing columns with prefix: ts018
Normalizing columns with prefix: ts007a
Normalizing columns with prefix: ts058
Normalizing columns with prefix: ts052
Normalizing columns with prefix: ts003
Normalizing columns with prefix: ts037
Normalizing columns with prefix: ts056
Normalizing columns with prefix: ts025
Normalizing columns with prefix: ts044
Normalizing columns with prefix: ts026
Normalizing columns with prefix: ts046
Normalizing columns with prefix: ts023
Normalizing columns with prefix: ts016
Normalizing columns with prefix: ts020
Normalizing columns with prefix: ts008
Normalizing columns with prefix: ts030
Normalizing columns with prefix: ts041
Normalizing columns with prefix: ts005
Normalizing columns with prefix: ts055
Normalizing columns with prefix: ts062
Normalizing columns with prefix: ts065
Normalizing columns with prefix: ts011
Normalizing columns with prefix: ts068
Normalizing columns with prefix: ts067
Normalizing columns with

In [ ]:
#find columns in eng_census_raw that are not in All_census
missing_columns = [col for col in eng_census_raw.columns if col not in All_census.columns]
print("Missing columns in All_census:")
print(missing_columns)

#and the other way around
missing_columns = [col for col in All_census.columns if col not in eng_census_raw.columns]
print("Missing columns in eng_census_raw:")
print(missing_columns)

#ts060 is "Industry" I dont think this is available any more at OA level
#ts004 is country of birth, i remove the duplicate variable Country of birth	Europe: Non-EU countries: All other non-EU countries


diff = eng_census_raw*100 - All_census
diff

Missing columns in All_census:
[]
Missing columns in eng_census_raw:
['ts0040015', 'ts0600002', 'ts0600003', 'ts0600004', 'ts0600005', 'ts0600006', 'ts0600007', 'ts0600008', 'ts0600009', 'ts0600010']


In [13]:
#print columns with nans
print("Columns with missing values:")
#in eng_census_raw
missing_columns = eng_census_raw.columns[eng_census_raw.isnull().any()].tolist()
print(missing_columns)

Columns with missing values:
['ts0200002', 'ts0200003', 'ts0550002', 'ts0550003', 'ts0550004', 'ts0550005', 'ts0550006', 'ts0550007', 'ts0550008', 'ts0550009', 'ts0550010']


In [14]:
#drop ts020 and ts055 (they are variables for small subsets (non-uk residents and second homes) so do not have values for most OAs
#drop columns starting with ts020 and ts055

columns_to_drop = [col for col in eng_census_raw.columns if col.startswith('ts020') or col.startswith('ts055')]
print("Dropping columns starting with 'ts020' or 'ts055':", columns_to_drop)
# Drop the columns
eng_census_raw = eng_census_raw.drop(columns=columns_to_drop)
#and the same for All_census
columns_to_drop = [col for col in All_census.columns if col.startswith('ts020') or col.startswith('ts055')]
print("Dropping columns starting with 'ts020' or 'ts055':", columns_to_drop)
# Drop the columns
All_census = All_census.drop(columns=columns_to_drop)


Dropping columns starting with 'ts020' or 'ts055': ['ts0200002', 'ts0200003', 'ts0550002', 'ts0550003', 'ts0550004', 'ts0550005', 'ts0550006', 'ts0550007', 'ts0550008', 'ts0550009', 'ts0550010']
Dropping columns starting with 'ts020' or 'ts055': ['ts0200002', 'ts0200003', 'ts0550002', 'ts0550003', 'ts0550004', 'ts0550005', 'ts0550006', 'ts0550007', 'ts0550008', 'ts0550009', 'ts0550010']


In [ ]:
def get_pca_filter_columns(All_census):

    # Filter columns with 'ts' in their names.
    ts_columns = [col for col in All_census.columns if 'ts' in col]
    ts_data = All_census[ts_columns]

    # Standardize the data.
    scaler = StandardScaler()
    ts_data_std = scaler.fit_transform(ts_data)

    # Apply PCA to account for 80% of the variance.
    pca = PCA(n_components=0.8)  # Adjusted to select components for 80% variance.
    pca.fit(ts_data_std)

    # The explained variance ratio indicates the importance of each principal component.
    explained_variance_ratio = pca.explained_variance_ratio_

    # Number of components chosen.
    num_components_chosen = pca.n_components_

    # Components_ gives the loadings of each variable on the principal components.
    loadings = pca.components_

    # Calculate the overall importance of each variable within the chosen components.
    # This is done by summing the squares of the loadings across the chosen components for each variable.
    overall_importance = np.sum(loadings**2, axis=0)

    # Identify the indices of the most important variables based on overall importance within the chosen components.
    important_indices = np.argsort(overall_importance)[::-1][:min(200, num_components_chosen)]

    # Extract the names of the most important variables.
    important_variables = np.array(ts_columns)[important_indices]

    return important_variables

def get_high_variance_columns(All_census,var_threshold=1.2):
    variances = {col: All_census[col].var() for col in All_census if col.startswith("ts")}

# Store variances in a new DataFrame
    var_df = pd.DataFrame(list(variances.items()), columns=['Column', 'Variance'])

# Find the median variance
#median_variance = All_census.var().median()  # this raises an error
    median_variance = var_df['Variance'].median() # I assume this is the aim, based on the comment and code above

# Determine 20% higher than the median variance
    threshold_variance = median_variance * var_threshold

# Subset the original DataFrame to include columns 20% higher than the median variance
    selected_columns = [col for col, var in variances.items() if var > threshold_variance]

    return selected_columns


In [22]:



important_variables = get_pca_filter_columns(All_census)
selected_columns = get_high_variance_columns(All_census)
#print All_census columns
print("PCA important variables:")
print(important_variables)
print("High variance columns:")
print(selected_columns)
print("All columns:")
print(All_census.columns.to_list())

# #  Select all the "important" variables based on PCA and varience
# concatenated_columns = np.unique(np.concatenate((important_variables, selected_columns)))
#only run varience prefileter
concatenated_columns = selected_columns
#dont have either
concatenated_columns = All_census.columns.to_list()


subset_df = All_census[concatenated_columns]
#subset_df

# Scale values [0, 1]
df_scaled = subset_df / 100.0

#round to 3dp 
df_scaled = df_scaled.round(3)
df_scaled = df_scaled.reset_index() #OA to col
# Save the scaled DataFrame to a CSV file
cleaned_data_path = "../data/engcensus_cleaned_scaled.parquet"
df_scaled.to_parquet(cleaned_data_path, index=False)
print(f"Scaled DataFrame saved to {cleaned_data_path}")

ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [5]:
#load the cleaned data
cleaned_data_path = "../data/engcensus_cleaned_scaled.parquet"
df_scaled = pd.read_parquet(cleaned_data_path)
df_scaled

,OA,ts0010002_PCT,ts0010003_PCT,ts0020002_PCT,ts0020003_PCT,ts0020004_PCT,ts0020005_PCT,ts0020006_PCT,ts0020007_PCT,ts0020008_PCT,...,ts0670007_PCT,ts0670008_PCT,ts0680002_PCT,ts0680003_PCT,ts0750002_PCT,ts0750003_PCT,ts0750004_PCT,ts0750005_PCT,ts0750006_PCT,ts0750007_PCT
0,E00060274,1.0,0.0,0.497,0.301,0.301,0.301,0.000,0.000,0.000,...,0.146,0.042,0.275,0.725,0.259,0.000,0.241,0.384,0.116,0.000
1,E00060275,1.0,0.0,0.508,0.344,0.331,0.325,0.007,0.013,0.000,...,0.184,0.016,0.251,0.749,0.217,0.025,0.280,0.325,0.153,0.000
2,E00060276,1.0,0.0,0.366,0.437,0.432,0.432,0.000,0.005,0.000,...,0.178,0.023,0.166,0.834,0.287,0.000,0.374,0.296,0.035,0.009
3,E00060277,1.0,0.0,0.464,0.368,0.368,0.368,0.000,0.000,0.000,...,0.163,0.014,0.245,0.755,0.336,0.008,0.295,0.238,0.123,0.000
4,E00060279,1.0,0.0,0.388,0.415,0.415,0.415,0.000,0.000,0.000,...,0.163,0.027,0.202,0.798,0.281,0.017,0.347,0.240,0.116,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188875,W00006938,1.0,0.0,0.240,0.542,0.542,0.542,0.000,0.000,0.000,...,0.396,0.005,0.160,0.840,0.333,0.010,0.275,0.216,0.167,0.000
188876,W00006940,1.0,0.0,0.302,0.509,0.503,0.500,0.003,0.006,0.006,...,0.273,0.038,0.181,0.819,0.254,0.034,0.277,0.288,0.147,0.000
188877,W00006941,1.0,0.0,0.393,0.363,0.363,0.355,0.009,0.000,0.000,...,0.197,0.047,0.139,0.861,0.326,0.031,0.209,0.279,0.147,0.008
188878,W00006942,1.0,0.0,0.479,0.335,0.335,0.335,0.000,0.000,0.000,...,0.160,0.043,0.240,0.760,0.269,0.023,0.169,0.415,0.115,0.008
